In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit

In [3]:
spark = get_sparkSession("Load bronze.customer")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "bronze"
_table_name = "customer"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "raw_customer.ipynb"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for bronze.customer


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from bronze.raw_customer_field and extracting data to customer table

df = spark.read.table("bronze.raw_customer_feed") \
          .select("customer_id", "store", "channel", "first_name", "middle_name", "last_name", "phone_number",
                  "email", "gender", "date_of_birth", "effective_start_dt", "is_current", "is_deleted", "source_file") \
          .withColumnRenamed("effective_start_dt","effective_start_date") \
          .dropDuplicates(["customer_id", "store", "channel", "effective_start_date"]) \
          .drop("effective_start_dt")

loaded_count = df.count()

In [9]:
df.show(10)
df.printSchema()
print(f"SPARK-APP : Rows entered : {loaded_count}")

+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+
|customer_id| store|channel|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|effective_start_date|is_current|is_deleted|     source_file|
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+
|    CUST001|AMZ_US|    MKT|      John|       null|      Doe|  9991112222|    john@gmail.com|  Male|   1988-05-10|          2024-01-01|     false|     false|dim_customer.csv|
|    CUST001|AMZ_US|    MKT|      John|       null|      Doe|  9991112222|john.doe@gmail.com|  Male|   1988-05-10|          2024-07-01|      true|     false|dim_customer.csv|
|    CUST002|  EBAY|    MKT|     Anita|       null|   Sharma|  8882223333|   anita@gmail.com|Female|   1992-03-18|          2

In [10]:

df_ld = df.withColumn("created_on", current_timestamp()) \
        .withColumn("created_by", lit(_script_name)) \
        .withColumn("modified_on", current_timestamp()) \
        .withColumn("modified_by", lit(_script_name))

In [11]:
df_ld.show(10)

+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+------------------+--------------------+------------------+
|customer_id| store|channel|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|effective_start_date|is_current|is_deleted|     source_file|          created_on|        created_by|         modified_on|       modified_by|
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+------------------+--------------------+------------------+
|    CUST001|AMZ_US|    MKT|      John|       null|      Doe|  9991112222|    john@gmail.com|  Male|   1988-05-10|          2024-01-01|     false|     false|dim_customer.csv|2026-01-29 00:02:...|raw_customer.ipynb|2026-01-29 00:02:...|

In [12]:
max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)

_data = []
if max_timestamp != '1900-01-01 00:00:00.000000':
    df_ld.write \
      .format("delta") \
      .mode("append") \
      .saveAsTable(_fullname)
    
    _data.append([_schema_name, _schema_name, _table_name, "delta-load", "append", str(datetime.now()), "success", loaded_count, _script_name, _script_name])
    insert_control_logs(spark, _data)
    
else:
    df_ld.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(_fullname)
    
    _data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])
    insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to bronze.customer and load_controller


In [13]:
spark.read.table("bronze.customer").show()

+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+------------------+--------------------+------------------+
|customer_id| store|channel|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|effective_start_date|is_current|is_deleted|     source_file|          created_on|        created_by|         modified_on|       modified_by|
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+------------------+--------------------+------------------+
|    CUST001|AMZ_US|    MKT|      John|       null|      Doe|  9991112222|    john@gmail.com|  Male|   1988-05-10|          2024-01-01|     false|     false|dim_customer.csv|2026-01-29 00:02:...|raw_customer.ipynb|2026-01-29 00:02:...|

In [14]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+------------------+--------------------+------------------+
| layer|schema_name|table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|        created_by|         modified_on|       modified_by|
+------+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+------------------+--------------------+------------------+
|bronze|     bronze|  customer|full-load| overwrite|2026-01-29 00:02:...|    success|           5|2026-01-29 00:02:...|raw_customer.ipynb|2026-01-29 00:02:...|raw_customer.ipynb|
+------+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+------------------+--------------------+------------------+



In [15]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |5            |
+-------+-------------+



In [16]:
spark.stop()